In [7]:
import cv2
import numpy as np
import math

def feature_extractor(img_url):
    # Load image
    img = cv2.imread(img_url)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Threshold to binary
    _, thresh = cv2.threshold(gray, 200, 255, cv2.THRESH_BINARY_INV)

    # Optional: clean noise
    kernel = np.ones((3,3), np.uint8)
    clean = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel)

    contours, _ = cv2.findContours(clean, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    rectangles = []
    for cnt in contours:
        approx = cv2.approxPolyDP(cnt, 0.02*cv2.arcLength(cnt, True), True)
        if len(approx) == 4:  # quadrilateral
            x,y,w,h = cv2.boundingRect(approx)
            rectangles.append((x,y,w,h))

    # Features
    areas = [w*h for (_,_,w,h) in rectangles]
    rect_count = len(rectangles)
    rect_coverage = sum(areas) / (img.shape[0]*img.shape[1])
    avg_rect_area = np.mean(areas) if areas else 0
    rect_stdev = np.std(areas) if areas else 0

    edges = cv2.Canny(clean, 50, 150, apertureSize=3)
    lines = cv2.HoughLinesP(edges, 1, np.pi/180, threshold=50, minLineLength=30, maxLineGap=5)

    line_lengths = []
    line_angles = []
    if lines is not None:
        for l in lines:
            x1,y1,x2,y2 = l[0]
            length = math.hypot(x2-x1, y2-y1)
            angle = math.degrees(math.atan2(y2-y1, x2-x1))
            line_lengths.append(length)
            line_angles.append(angle)

    # Features
    avg_line_length = np.mean(line_lengths) if line_lengths else 0
    longest_line = np.max(line_lengths) if line_lengths else 0
    shortest_line = np.min(line_lengths) if line_lengths else 0
    line_length_stdev = np.std(line_lengths) if line_lengths else 0
    avg_line_angle = np.mean(line_angles) if line_angles else 0

    # Orthogonality: ratio of lines near 0°/90°/180°
    orth_count = sum(1 for a in line_angles if abs(a % 90) < 5)
    orth_ratio = orth_count / len(line_angles) if line_angles else 0

    # Crossings: check intersections between line segments
    def intersect(l1, l2):
        # simple segment intersection test
        (x1,y1,x2,y2) = l1
        (x3,y3,x4,y4) = l2
        # vector cross product approach
        def ccw(A,B,C): return (C[1]-A[1])*(B[0]-A[0]) > (B[1]-A[1])*(C[0]-A[0])
        return ccw((x1,y1),(x3,y3),(x4,y4)) != ccw((x2,y2),(x3,y3),(x4,y4)) and \
            ccw((x1,y1),(x2,y2),(x3,y3)) != ccw((x1,y1),(x2,y2),(x4,y4))

    crossings = 0
    if lines is not None:
        segs = [tuple(l[0]) for l in lines]
        for i in range(len(segs)):
            for j in range(i+1, len(segs)):
                if intersect(segs[i], segs[j]):
                    crossings += 1
    bends = []
    for cnt in contours:
        approx = cv2.approxPolyDP(cnt, 0.02*cv2.arcLength(cnt, True), True)
        if len(approx) > 2:
            bends.append(len(approx))  # number of bends
    avg_line_bends = np.mean(bends) if bends else 0

    crossing_angles = []
    if lines is not None:
        segs = [tuple(l[0]) for l in lines]
        for i in range(len(segs)):
            for j in range(i+1, len(segs)):
                if intersect(segs[i], segs[j]):
                    (x1,y1,x2,y2) = segs[i]
                    (x3,y3,x4,y4) = segs[j]
                    a1 = math.atan2(y2-y1, x2-x1)
                    a2 = math.atan2(y4-y3, x4-x3)
                    crossing_angles.append(abs(a1-a2))
    avg_crossing_angle = np.mean(crossing_angles) if crossing_angles else 0


    dists = []
    for i in range(len(rectangles)):
        for j in range(i+1, len(rectangles)):
            (x1,y1,w1,h1) = rectangles[i]
            (x2,y2,w2,h2) = rectangles[j]
            # center points
            c1 = (x1+w1/2, y1+h1/2)
            c2 = (x2+w2/2, y2+h2/2)
            dists.append(math.hypot(c2[0]-c1[0], c2[1]-c1[1]))
    avg_shortest_distance = np.mean(dists) if dists else 0
    centers = [(x+w/2, y+h/2) for (x,y,w,h) in rectangles]
    if centers:
        cx = [c[0] for c in centers]
        cy = [c[1] for c in centers]
        rect_distribution = np.var(cx) + np.var(cy)
    else:
        rect_distribution = 0
        orth_count = 0
    for (x,y,w,h) in rectangles:
        aspect = w/h if h>0 else 0
        if abs(aspect-1) < 0.1:  # nearly square
            orth_count += 1
    rect_orth_count = 0
    for (x,y,w,h) in rectangles:
        aspect = w/h if h>0 else 0
        if abs(aspect-1) < 0.1:  # nearly square
            rect_orth_count += 1
    rect_orth = rect_orth_count / rect_count if rect_count else 0


    # RectOrth2: maybe alignment of rectangle centers along grid lines
    if centers:
        mean_x = np.mean([c[0] for c in centers])
        aligned = sum(1 for c in centers if abs(c[0]-mean_x) < 5)
        rect_orth2 = aligned / rect_count
    else:
        rect_orth2 = 0

    features = {
        "RectCoverage": rect_coverage,
        "AvgRectArea": avg_rect_area,
        "RectStDev": rect_stdev,
        "AspectRatio": np.mean([w/h for (_,_,w,h) in rectangles if h>0]) if rectangles else 0,
        "AvgLineBends": avg_line_bends,
        "AvgLineLength": avg_line_length,
        "LongestLine": longest_line,
        "ShortestLine": shortest_line,
        "LineLengthStDev": line_length_stdev,
        "AvgLineAngle": avg_line_angle,
        "OrthLinesRatio": orth_ratio,
        "LineCrossings": crossings,
        "AvgCrossingAngle": avg_crossing_angle,
        "AvgShortestDistance": avg_shortest_distance,
        "RectDistribution": rect_distribution,
        "RectOrth": rect_orth,
        "RectOrth2": rect_orth2,
        "rectangles": rect_count,
        "lines": len(lines) if lines is not None else 0
    }
    return features

In [8]:
import os
import pandas as pd
import cairosvg

def process_folder(folder_name, label, save_png=True):
    data = []
    # create a single "png" folder in the current working directory
    png_folder = os.path.join(os.getcwd(), "png")
    if save_png and not os.path.exists(png_folder):
        os.makedirs(png_folder)

    for file in os.listdir(folder_name):
        if file.endswith(".svg"):
            svg_path = os.path.join(folder_name, file)

            # --- Convert SVG → PNG ---
            if save_png:
                # save all PNGs into ./png/ with source label prefix
                png_name = f"{label}_{file.replace('.svg', '.png')}"
                png_path = os.path.join(png_folder, png_name)
                cairosvg.svg2png(url=svg_path, write_to=png_path)

            # preprocess data
            feats = feature_extractor(png_path)
            feats["diagram_name"] = file
            feats["source"] = label
            data.append(feats)
    return pd.DataFrame(data)

# Process all folders
df_ground = process_folder("../../Graph Generation/Ground_truth/SVG", "GroundTruth")
df_llm_a = process_folder("../../Graph Generation/generated_svgs/claude_4_5", "claude_4_5")
df_llm_b = process_folder("../../Graph Generation/generated_svgs/gpt4o", "gpt4o")
df_llm_c = process_folder("../../Graph Generation/generated_svgs/gpt5", "gpt5")

# Combine
df_all = pd.concat([df_ground, df_llm_a, df_llm_b, df_llm_c], ignore_index=True)

# Save
df_all.to_csv("all_features.csv", index=False)